# 04 — Statistical Analysis

## But First, Coffee: Competitive Revenue Intelligence

This notebook applies inferential statistical analysis only where the structure and quality of the collected data support valid hypothesis testing.

The primary inferential analysis evaluates whether regular menu prices differ systematically across But First, Coffee, Starbucks, and The Coffee Bean & Tea Leaf using directly matched product families.

Statistical significance is interpreted together with effect size, sample limitations, and business relevance.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

ALPHA = 0.05

PROJECT_ROOT = Path.cwd().parent

DATA_CLEANED = (
    PROJECT_ROOT
    / "data"
    / "cleaned"
)

TABLES = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
)

FIGURES = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
)

print("Project root:", PROJECT_ROOT)
print("Alpha:", ALPHA)

Project root: /Users/jannoelvero/Desktop/but_first_coffee_revenue_analysis
Alpha: 0.05


In [2]:
matched_prices = pd.read_csv(
    DATA_CLEANED
    / "matched_family_prices.csv"
)

print(
    "Shape:",
    matched_prices.shape
)

matched_prices

Shape: (15, 4)


,brand,comparable_product_family,representative_price_php,source_observations
0,"But First, Coffee",Americano,130.0,1
1,"But First, Coffee",Cafe Latte,150.0,1
2,"But First, Coffee",Caramel Macchiato,130.0,3
3,"But First, Coffee",Matcha Latte,130.0,3
4,"But First, Coffee",Mocha,150.0,1
5,Starbucks,Americano,175.0,2
6,Starbucks,Cafe Latte,185.0,2
7,Starbucks,Caramel Macchiato,210.0,2
8,Starbucks,Matcha Latte,190.0,2
9,Starbucks,Mocha,205.0,2


In [4]:
print("Shape:", matched_prices.shape)

print("\nColumns:")
for col in matched_prices.columns:
    print(repr(col))

print("\nFirst 10 rows:")
display(matched_prices.head(10))

Shape: (15, 4)

Columns:
'brand'
'comparable_product_family'
'representative_price_php'
'source_observations'

First 10 rows:


,brand,comparable_product_family,representative_price_php,source_observations
0,"But First, Coffee",Americano,130.0,1
1,"But First, Coffee",Cafe Latte,150.0,1
2,"But First, Coffee",Caramel Macchiato,130.0,3
3,"But First, Coffee",Matcha Latte,130.0,3
4,"But First, Coffee",Mocha,150.0,1
5,Starbucks,Americano,175.0,2
6,Starbucks,Cafe Latte,185.0,2
7,Starbucks,Caramel Macchiato,210.0,2
8,Starbucks,Matcha Latte,190.0,2
9,Starbucks,Mocha,205.0,2


## Friedman Test — Matched Product-Family Pricing

### Research Question

Do regular menu prices differ systematically among But First, Coffee, Starbucks, and The Coffee Bean & Tea Leaf when directly comparable product families are evaluated?

### Null Hypothesis (H₀)

There is no systematic difference in price ranks among the three coffee brands across the matched product families.

### Alternative Hypothesis (H₁)

At least one brand has a systematically different price rank across the matched product families.

### Significance Level

\[
\alpha = 0.05
\]

### Decision Rule

Reject \(H_0\) if:

\[
p < 0.05
\]

For the asymptotic Friedman test with three brands:

\[
df = k - 1 = 3 - 1 = 2
\]

The corresponding chi-square critical value at \(\alpha = 0.05\) is approximately:

\[
\chi^2_{0.05,2} = 5.991
\]

## Test Selection

The Friedman test is used because:

1. the same five product families are observed across all three brands;
2. the observations therefore form matched blocks rather than independent groups;
3. the number of matched product families is small;
4. a rank-based non-parametric repeated-measures comparison avoids relying on a normality assumption for such a small matched sample.

The product family is treated as the blocking variable, while brand is the treatment/comparison factor.

The analysis intentionally avoids applying ANOVA or Kruskal–Wallis to all menu observations because differences in product and category composition would confound the competitive price comparison.

In [14]:
matched_design_check = (
    matched_prices
    .groupby(
        "comparable_product_family"
    )["brand"]
    .nunique()
)

print("Brands per product family:")
display(matched_design_check)

print(
    "\nProduct families:",
    matched_prices[
        "comparable_product_family"
    ].nunique()
)

print(
    "Brands:",
    matched_prices[
        "brand"
    ].nunique()
)

print(
    "Complete blocks:",
    (
        matched_design_check == 3
    ).sum()
)

Brands per product family:


comparable_product_family
Americano            3
Cafe Latte           3
Caramel Macchiato    3
Matcha Latte         3
Mocha                3
Name: brand, dtype: int64


Product families: 5
Brands: 3
Complete blocks: 5


In [15]:
price_matrix = (
    matched_prices
    .pivot(
        index="comparable_product_family",
        columns="brand",
        values="representative_price_php"
    )
)

price_matrix

brand,"But First, Coffee",Starbucks,The Coffee Bean & Tea Leaf
comparable_product_family,,,
Americano,130.0,175.0,185.0
Cafe Latte,150.0,185.0,187.5
Caramel Macchiato,130.0,210.0,245.0
Matcha Latte,130.0,190.0,225.0
Mocha,150.0,205.0,220.0


In [20]:
price_ranks = (
    price_matrix
    .rank(
        axis=1,
        method="average"
    )
)

price_ranks

brand,"But First, Coffee",Starbucks,The Coffee Bean & Tea Leaf
comparable_product_family,,,
Americano,1.0,2.0,3.0
Cafe Latte,1.0,2.0,3.0
Caramel Macchiato,1.0,2.0,3.0
Matcha Latte,1.0,2.0,3.0
Mocha,1.0,2.0,3.0


In [21]:
mean_price_ranks = (
    price_ranks
    .mean()
    .sort_values()
)

mean_price_ranks

brand
But First, Coffee             1.0
Starbucks                     2.0
The Coffee Bean & Tea Leaf    3.0
dtype: float64

In [22]:
friedman_statistic, friedman_pvalue = (
    stats.friedmanchisquare(
        price_matrix[
            "But First, Coffee"
        ],
        price_matrix[
            "Starbucks"
        ],
        price_matrix[
            "The Coffee Bean & Tea Leaf"
        ]
    )
)

print(
    f"Friedman statistic: "
    f"{friedman_statistic:.4f}"
)

print(
    f"p-value: "
    f"{friedman_pvalue:.6f}"
)

print(
    f"Alpha: {ALPHA}"
)

Friedman statistic: 10.0000
p-value: 0.006738
Alpha: 0.05


In [23]:
k = price_matrix.shape[1]

df = k - 1

critical_value = stats.chi2.ppf(
    1 - ALPHA,
    df
)

print(
    f"Degrees of freedom: {df}"
)

print(
    f"Critical chi-square value: "
    f"{critical_value:.4f}"
)

print(
    f"Observed Friedman statistic: "
    f"{friedman_statistic:.4f}"
)

if friedman_statistic > critical_value:
    print(
        "Decision: Reject H0"
    )
else:
    print(
        "Decision: Fail to reject H0"
    )

Degrees of freedom: 2
Critical chi-square value: 5.9915
Observed Friedman statistic: 10.0000
Decision: Reject H0


In [24]:
n_blocks = price_matrix.shape[0]

kendalls_w = (
    friedman_statistic
    / (
        n_blocks
        * (k - 1)
    )
)

print(
    f"Number of matched families: "
    f"{n_blocks}"
)

print(
    f"Kendall's W: "
    f"{kendalls_w:.4f}"
)

Number of matched families: 5
Kendall's W: 1.0000


## Friedman Test Result

The Friedman test identified a statistically significant difference in representative price ranks among But First, Coffee, Starbucks, and The Coffee Bean & Tea Leaf across the five matched product families:

\[
Q(2) = 10.000,\quad p = 0.006738
\]

At the 5% significance level:

\[
p < 0.05
\]

and the observed test statistic exceeded the chi-square critical value:

\[
10.000 > 5.991
\]

Therefore, the null hypothesis is rejected.

The mean ranks were:

- But First, Coffee = 1.00
- Starbucks = 2.00
- The Coffee Bean & Tea Leaf = 3.00

Kendall's coefficient of concordance was:

\[
W = 1.00
\]

indicating complete consistency in the relative price ranking across the five matched product families in this sample.

The result establishes an overall systematic difference in price ranks across the three brands. It does not, by itself, establish which individual pairwise brand comparisons are statistically significant.

## Business Interpretation

The statistical result provides evidence that But First, Coffee's lower-price positioning is not driven by only one isolated menu item.

Across all five directly comparable product families in the matched sample, But First, Coffee had the lowest representative price, Starbucks occupied the middle position, and The Coffee Bean & Tea Leaf had the highest representative price.

Combined with the exploratory pricing analysis, this supports the interpretation that affordability is a consistent component of But First, Coffee's observed competitive positioning.

However, statistical evidence of lower prices is not evidence that But First, Coffee should broadly increase prices.

A revenue-management pricing decision would additionally require internal evidence on:

- transaction volume;
- product mix;
- price elasticity;
- promotional response;
- product-level variable cost;
- contribution margin;
- repeat-purchase behavior; and
- customer response to price changes.

The external analysis therefore identifies pricing headroom as a hypothesis for controlled testing rather than recommending an immediate across-the-board price increase.

## Statistical Limitation

The Friedman analysis is based on five matched product families.

Although the observed ranking is perfectly consistent and the omnibus Friedman test is statistically significant, the number of matched blocks is small.

The result should therefore be interpreted as strong evidence within the selected directly comparable product families rather than as proof that every product sold by the three brands follows the same pricing relationship.

The matched-family approach improves comparability by controlling for product-family composition, but broader menu coverage would strengthen external validity.

In [25]:
pairwise_differences = pd.DataFrame({
    "product_family": price_matrix.index,

    "BFC_vs_Starbucks_php": (
        price_matrix["But First, Coffee"]
        - price_matrix["Starbucks"]
    ).values,

    "BFC_vs_CBTL_php": (
        price_matrix["But First, Coffee"]
        - price_matrix[
            "The Coffee Bean & Tea Leaf"
        ]
    ).values,

    "Starbucks_vs_CBTL_php": (
        price_matrix["Starbucks"]
        - price_matrix[
            "The Coffee Bean & Tea Leaf"
        ]
    ).values
})

pairwise_differences

,product_family,BFC_vs_Starbucks_php,BFC_vs_CBTL_php,Starbucks_vs_CBTL_php
0,Americano,-45.0,-55.0,-10.0
1,Cafe Latte,-35.0,-37.5,-2.5
2,Caramel Macchiato,-80.0,-115.0,-35.0
3,Matcha Latte,-60.0,-95.0,-35.0
4,Mocha,-55.0,-70.0,-15.0


In [26]:
pairwise_gap_summary = (
    pairwise_differences
    .drop(
        columns="product_family"
    )
    .agg([
        "mean",
        "median",
        "min",
        "max"
    ])
    .T
    .round(2)
)

pairwise_gap_summary

,mean,median,min,max
BFC_vs_Starbucks_php,-55.0,-55.0,-80.0,-35.0
BFC_vs_CBTL_php,-74.5,-70.0,-115.0,-37.5
Starbucks_vs_CBTL_php,-19.5,-15.0,-35.0,-2.5


## Post-hoc Pairwise Analysis

Because the omnibus Friedman test detected a statistically significant difference among the three brands, exploratory post-hoc pairwise comparisons are conducted using the Wilcoxon signed-rank test.

Each comparison uses the same five matched product families.

### Pairwise Null Hypothesis

For each pair of brands:

\[
H_0:
\text{The distribution of paired price differences is centered at zero.}
\]

### Pairwise Alternative Hypothesis

\[
H_1:
\text{The distribution of paired price differences is not centered at zero.}
\]

Because three pairwise comparisons are conducted, the family-wise Type I error rate is controlled using the Bonferroni correction:

\[
\alpha_{\text{adjusted}}
=
\frac{0.05}{3}
=
0.0167
\]

The very small number of matched product families (\(n=5\)) substantially limits the statistical power and attainable p-values of the pairwise tests.

In [27]:
pairs = [
    (
        "But First, Coffee",
        "Starbucks"
    ),
    (
        "But First, Coffee",
        "The Coffee Bean & Tea Leaf"
    ),
    (
        "Starbucks",
        "The Coffee Bean & Tea Leaf"
    )
]

pairwise_results = []

for brand_1, brand_2 in pairs:

    statistic, p_value = stats.wilcoxon(
        price_matrix[brand_1],
        price_matrix[brand_2],
        alternative="two-sided",
        method="exact"
    )

    pairwise_results.append({
        "comparison":
            f"{brand_1} vs {brand_2}",
        "wilcoxon_statistic":
            statistic,
        "raw_p_value":
            p_value
    })

pairwise_results = pd.DataFrame(
    pairwise_results
)

pairwise_results

,comparison,wilcoxon_statistic,raw_p_value
0,"But First, Coffee vs Starbucks",0.0,0.0625
1,"But First, Coffee vs The Coffee Bean & Tea Leaf",0.0,0.0625
2,Starbucks vs The Coffee Bean & Tea Leaf,0.0,0.0625


In [28]:
n_comparisons = len(
    pairwise_results
)

bonferroni_alpha = (
    ALPHA
    / n_comparisons
)

pairwise_results[
    "bonferroni_p_value"
] = np.minimum(
    pairwise_results[
        "raw_p_value"
    ]
    * n_comparisons,
    1.0
)

pairwise_results[
    "significant_after_bonferroni"
] = (
    pairwise_results[
        "bonferroni_p_value"
    ]
    < ALPHA
)

print(
    f"Number of comparisons: "
    f"{n_comparisons}"
)

print(
    f"Bonferroni-adjusted alpha: "
    f"{bonferroni_alpha:.4f}"
)

display(
    pairwise_results.round(4)
)

Number of comparisons: 3
Bonferroni-adjusted alpha: 0.0167


,comparison,wilcoxon_statistic,raw_p_value,bonferroni_p_value,significant_after_bonferroni
0,"But First, Coffee vs Starbucks",0.0,0.0625,0.1875,False
1,"But First, Coffee vs The Coffee Bean & Tea Leaf",0.0,0.0625,0.1875,False
2,Starbucks vs The Coffee Bean & Tea Leaf,0.0,0.0625,0.1875,False


In [29]:
pairwise_business_summary = pd.DataFrame({
    "comparison": [
        "BFC vs Starbucks",
        "BFC vs CBTL",
        "Starbucks vs CBTL"
    ],

    "mean_difference_php": [
        pairwise_differences[
            "BFC_vs_Starbucks_php"
        ].mean(),

        pairwise_differences[
            "BFC_vs_CBTL_php"
        ].mean(),

        pairwise_differences[
            "Starbucks_vs_CBTL_php"
        ].mean()
    ],

    "median_difference_php": [
        pairwise_differences[
            "BFC_vs_Starbucks_php"
        ].median(),

        pairwise_differences[
            "BFC_vs_CBTL_php"
        ].median(),

        pairwise_differences[
            "Starbucks_vs_CBTL_php"
        ].median()
    ],

    "direction_consistency": [
        "BFC lower in 5/5",
        "BFC lower in 5/5",
        "Starbucks lower in 5/5"
    ]
})

pairwise_business_summary.round(2)

,comparison,mean_difference_php,median_difference_php,direction_consistency
0,BFC vs Starbucks,-55.0,-55.0,BFC lower in 5/5
1,BFC vs CBTL,-74.5,-70.0,BFC lower in 5/5
2,Starbucks vs CBTL,-19.5,-15.0,Starbucks lower in 5/5


## Post-hoc Pairwise Interpretation

The omnibus Friedman test established a statistically significant systematic difference in price ranks among the three brands.

Exploratory pairwise Wilcoxon signed-rank tests were subsequently conducted using the five matched product families.

For all three comparisons, the Wilcoxon statistic was:

\[
W = 0
\]

and the exact two-sided p-value was:

\[
p = 0.0625
\]

After Bonferroni correction for three pairwise comparisons:

\[
p_{\text{adjusted}} = 0.1875
\]

Therefore, none of the individual pairwise comparisons reached statistical significance at the family-wise 5% significance level.

This does not contradict the significant Friedman result. With only five matched product families, the exact two-sided Wilcoxon test has limited statistical resolution and power. Even though every observed paired difference followed the same direction, the available number of matched pairs was insufficient for the individual two-sided comparisons to reach conventional statistical significance.

The descriptive magnitude remains commercially relevant:

- But First, Coffee was lower-priced than Starbucks in 5 of 5 matched families, with an average observed difference of ₱55.
- But First, Coffee was lower-priced than The Coffee Bean & Tea Leaf in 5 of 5 matched families, with an average observed difference of ₱74.50.
- Starbucks was lower-priced than The Coffee Bean & Tea Leaf in 5 of 5 matched families, with an average observed difference of ₱19.50.

Accordingly, the evidence supports a systematic overall price-positioning difference across the three brands, while individual pairwise inferential conclusions remain limited by the small number of matched product families.

In [33]:
individual_ratings = pd.read_csv(
    DATA_CLEANED
    / "individual_ratings_cleaned.csv"
)

print(
    "Individual ratings shape:",
    individual_ratings.shape
)

print("\nColumns:")
print(
    individual_ratings.columns.tolist()
)

display(
    individual_ratings.head()
)

Individual ratings shape: (13, 9)

Columns:
['brand', 'branch', 'platform_origin', 'rating', 'review_recency', 'review_text_paraphrase', 'source_page', 'source_type', 'verification_note']


,brand,branch,platform_origin,rating,review_recency,review_text_paraphrase,source_page,source_type,verification_note
0,"But First, Coffee",Lipa Tambo,Google,5,1 year ago,"Customer praised coffee, staff, ambience and s...",https://www.top-rated.online/cities/Lipa/place...,Google-origin review indexed by Top-Rated.Online,Individual 5/5 visibly attached to review
1,"But First, Coffee",Lipa Tambo,Google,4,3 months ago,Customer described the coffee as delicious.,https://www.top-rated.online/cities/Lipa/place...,Google-origin review indexed by Top-Rated.Online,Individual 4/5 visibly attached to review
2,"But First, Coffee",Lipa Tambo,Google,5,1 year ago,"Customer praised drinks, non-dairy options, st...",https://www.top-rated.online/cities/Lipa/place...,Google-origin review indexed by Top-Rated.Online,Individual 5/5 visibly attached to review
3,"But First, Coffee",Lipa Tambo,Google,5,2 years ago,"Customer highlighted affordable menu, accessib...",https://www.top-rated.online/cities/Lipa/place...,Google-origin review indexed by Top-Rated.Online,Individual 5/5 visibly attached to review
4,"But First, Coffee",Lipa Tambo,Google,5,1 year ago,"Customer liked Vietnamese coffee, pesto pasta ...",https://www.top-rated.online/cities/Lipa/place...,Google-origin review indexed by Top-Rated.Online,Individual 5/5 visibly attached to review


In [34]:
individual_rating_availability = (
    individual_ratings
    .groupby("brand")
    .size()
    .reindex([
        "But First, Coffee",
        "Starbucks",
        "The Coffee Bean & Tea Leaf"
    ], fill_value=0)
    .rename("individual_ratings")
)

individual_rating_availability

brand
But First, Coffee             13
Starbucks                      0
The Coffee Bean & Tea Leaf     0
Name: individual_ratings, dtype: int64

In [35]:
rq2_eligible = (
    (
        individual_rating_availability
        > 0
    ).all()
)

print(
    "RQ2 cross-brand hypothesis test eligible:",
    rq2_eligible
)

if not rq2_eligible:
    print(
        "Reason: comparable individual-level ratings "
        "are unavailable for all three brands."
    )

RQ2 cross-brand hypothesis test eligible: False
Reason: comparable individual-level ratings are unavailable for all three brands.


## RQ2 — Statistical Test Eligibility

RQ2 asks whether individual customer ratings differ significantly among But First, Coffee, Starbucks, and The Coffee Bean & Tea Leaf.

A valid cross-brand hypothesis test requires comparable individual-level rating observations for all brands.

The collected dataset contains 13 verified individual ratings for But First, Coffee but no comparable individual-level ratings for Starbucks or The Coffee Bean & Tea Leaf.

Therefore:

\[
\text{RQ2 inferential test eligibility} = \text{False}
\]

No ANOVA, Kruskal–Wallis test, t-test, Mann–Whitney test, or other cross-brand rating hypothesis test is conducted.

Branch/platform aggregate ratings are not substituted for individual customer ratings because aggregate ratings summarize multiple customers and represent a different unit of analysis.

Repeatedly assigning a branch aggregate rating to individual written-review rows would create pseudoreplication and artificially inflate the effective sample size.

RQ2 therefore remains descriptively documented but inferentially unresolved with the available external data.

In [36]:
bfc_individual_rating_summary = pd.DataFrame({
    "metric": [
        "Individual ratings",
        "Mean rating",
        "Median rating",
        "Minimum rating",
        "Maximum rating",
        "Five-star ratings",
        "Five-star share"
    ],
    "value": [
        len(individual_ratings),
        round(
            individual_ratings[
                "rating"
            ].mean(),
            2
        ),
        round(
            individual_ratings[
                "rating"
            ].median(),
            2
        ),
        individual_ratings[
            "rating"
        ].min(),
        individual_ratings[
            "rating"
        ].max(),
        (
            individual_ratings[
                "rating"
            ] == 5
        ).sum(),
        f"{(
            individual_ratings['rating']
            .eq(5)
            .mean()
            * 100
        ):.2f}%"
    ]
})

bfc_individual_rating_summary

,metric,value
0,Individual ratings,13
1,Mean rating,4.69
2,Median rating,5.0
3,Minimum rating,2
4,Maximum rating,5
5,Five-star ratings,11
6,Five-star share,84.62%


## Individual Rating Descriptive Evidence

The available individual-level evidence for But First, Coffee contains 13 verified ratings.

Within this sample:

- the mean rating was approximately 4.69;
- the median rating was 5.00;
- 11 of 13 observations were five-star ratings; and
- the observed five-star share was 84.62%.

These results indicate strongly positive ratings within the collected But First, Coffee sample.

However, the sample is small, geographically limited, and based on externally discoverable Google-origin evidence. It may therefore be affected by selection and visibility bias.

The result should not be interpreted as an estimate of the population-wide customer satisfaction rate for But First, Coffee, nor can it be used to establish that But First, Coffee has higher customer satisfaction than Starbucks or The Coffee Bean & Tea Leaf.

In [37]:
reviews = pd.read_csv(
    DATA_CLEANED
    / "reviews_cleaned.csv"
)

print(
    "Cleaned reviews shape:",
    reviews.shape
)

print("\nFirst 20 columns:")
print(
    reviews.columns.tolist()[:20]
)

Cleaned reviews shape: (282, 52)

First 20 columns:
['brand', 'branch', 'platform', 'aggregate_rating', 'rating_volume', 'review_date', 'review_text', 'initial_theme_hint', 'source_url', 'review_text_original', 'theme_original', 'review_text_clean', 'theme_list', 'review_date_parsed', 'theme_standard_list', 'theme_add_on', 'theme_availability', 'theme_coffee_strength', 'theme_consistency', 'theme_customization']


In [38]:
print(
    "Total review observations:",
    len(reviews)
)

print(
    "Brands:",
    reviews["brand"].nunique()
)

print("\nReviews by brand:")
display(
    reviews[
        "brand"
    ]
    .value_counts()
)

print(
    "\nUnique branches by brand:"
)

display(
    reviews
    .groupby("brand")
    ["branch"]
    .nunique()
)

Total review observations: 282
Brands: 3

Reviews by brand:


brand
Starbucks                     108
But First, Coffee             104
The Coffee Bean & Tea Leaf     70
Name: count, dtype: int64


Unique branches by brand:


brand
But First, Coffee             19
Starbucks                     17
The Coffee Bean & Tea Leaf    19
Name: branch, dtype: int64

In [39]:
rq3_inferential_eligible = False

rq3_reasons = [
    "Reviews were collected from externally observable platform evidence rather than a probability sample.",
    "Multiple reviews originate from the same branches, creating potential within-branch dependence.",
    "A review can contain multiple customer-experience themes, so theme indicators are not mutually exclusive.",
    "The conservative polarity classifier identified explicit polarity for only 52.13% of the review sample.",
    "The polarity rules are analytical heuristics rather than a validated sentiment model."
]

print(
    "RQ3 population-level inferential test eligible:",
    rq3_inferential_eligible
)

print("\nReasons:")

for i, reason in enumerate(
    rq3_reasons,
    start=1
):
    print(
        f"{i}. {reason}"
    )

RQ3 population-level inferential test eligible: False

Reasons:
1. Reviews were collected from externally observable platform evidence rather than a probability sample.
2. Multiple reviews originate from the same branches, creating potential within-branch dependence.
3. A review can contain multiple customer-experience themes, so theme indicators are not mutually exclusive.
4. The conservative polarity classifier identified explicit polarity for only 52.13% of the review sample.
5. The polarity rules are analytical heuristics rather than a validated sentiment model.


## RQ3 — Statistical Test Eligibility

RQ3 asks which customer-experience factors are most frequently associated with positive and negative customer experiences across the three coffee brands.

The available review dataset supports descriptive association analysis but does not provide a sufficiently strong basis for population-level inferential claims.

Several characteristics of the data require caution:

1. the reviews represent externally observable platform evidence rather than a probability sample of customers;
2. multiple observations originate from the same branches, creating potential within-branch dependence;
3. individual reviews can contain multiple themes, meaning the theme indicators are not mutually exclusive;
4. the conservative polarity rules classified explicit positive, negative, or mixed evidence for only 52.13% of the total review sample; and
5. the polarity method is a transparent rule-based analytical procedure rather than a validated sentiment model.

For these reasons, no population-level chi-square test or similar inferential comparison is used as the primary answer to RQ3.

RQ3 is instead answered using descriptive theme prevalence, management-dimension prevalence, polarity-associated themes, and theme co-occurrence.

The findings describe patterns in the collected review evidence and should not be interpreted as causal relationships or population-wide customer satisfaction estimates.

In [40]:
top_negative_themes = pd.read_csv(
    TABLES
    / "eda_top_negative_themes.csv"
)

top_positive_themes = pd.read_csv(
    TABLES
    / "eda_top_positive_themes.csv"
)

dimension_by_polarity = pd.read_csv(
    TABLES
    / "eda_dimension_by_polarity.csv"
)

theme_cooccurrence = pd.read_csv(
    TABLES
    / "eda_theme_cooccurrence.csv"
)

print("Top negative themes:")
display(
    top_negative_themes.head(10)
)

print("\nTop positive themes:")
display(
    top_positive_themes.head(10)
)

print("\nDimension by polarity:")
display(
    dimension_by_polarity
)

print("\nTop theme co-occurrences:")
display(
    theme_cooccurrence.head(10)
)

Top negative themes:


,theme,negative_prevalence_pct
0,Taste,23.85
1,Packaging,21.10
2,Missing Item,20.18
3,Customization,11.93
4,Order Accuracy,11.01
5,Value,7.34
6,Portion,6.42
7,Size Accuracy,5.50
8,Add On,4.59
9,Food Quality,3.67



Top positive themes:


,theme,positive_prevalence_pct
0,Taste,57.14
1,Service,20.00
2,Consistency,17.14
3,Packaging,17.14
4,Customization,14.29
5,Loyalty,8.57
6,Speed,5.71
7,Positive,5.71
8,Value,2.86
9,Occasion,2.86



Dimension by polarity:


,Unnamed: 0,Mixed,Negative,Positive
0,Product Quality,66.67,26.61,60.00
1,Order Execution,0.00,36.70,0.00
2,Customization Addons,33.33,16.51,14.29
3,Packaging,0.00,21.10,17.14
4,Value Portion,33.33,10.09,2.86
5,Service Delivery,0.00,0.92,25.71
6,Availability Consistency,33.33,5.50,17.14
7,Loyalty Experience,0.00,0.00,17.14



Top theme co-occurrences:


,theme_1,theme_2,cooccurrence_count
0,Consistency,Taste,17
1,Portion,Value,8
2,Taste,Value,7
3,Customization,Service,5
4,Customization,Taste,5
5,Portion,Taste,4
6,Packaging,Taste,4
7,Quality,Taste,3
8,Packaging,Service,3
9,Order Accuracy,Taste,3


## RQ3 — Customer Experience Interpretation

The descriptive review analysis identifies product execution and operational execution as the principal customer-experience themes within the collected evidence.

Among reviews where the conservative polarity rules identified negative evidence, the most frequently associated themes were:

1. Taste — 23.85%
2. Packaging — 21.10%
3. Missing Item — 20.18%
4. Customization — 11.93%
5. Order Accuracy — 11.01%

At the broader management-dimension level, Order Execution was associated with 36.70% of classified negative experiences, followed by Product Quality at 26.61%.

Among reviews where positive evidence was identified, the leading themes were:

1. Taste — 57.14%
2. Service — 20.00%
3. Consistency — 17.14%
4. Packaging — 17.14%
5. Customization — 14.29%

Product Quality was associated with 60.00% of classified positive experiences, while Service & Delivery accounted for 25.71%.

Taste therefore appears prominently in both positive and negative experiences. The strongest theme co-occurrence was Taste + Consistency, appearing in 17 reviews.

From a revenue-management perspective, these patterns indicate that price and promotion decisions should not be evaluated independently from product consistency and order execution. Customer response to pricing may depend partly on whether the underlying product and service experience is delivered consistently.

These are descriptive associations within the collected review evidence and do not establish causality or population-level customer satisfaction differences.

In [41]:
# Remove accidental CSV index column if present
dimension_by_polarity = (
    dimension_by_polarity
    .drop(
        columns=["Unnamed: 0"],
        errors="ignore"
    )
)

dimension_by_polarity

,Mixed,Negative,Positive
0,66.67,26.61,60.00
1,0.00,36.70,0.00
2,33.33,16.51,14.29
3,0.00,21.10,17.14
4,33.33,10.09,2.86
5,0.00,0.92,25.71
6,33.33,5.50,17.14
7,0.00,0.00,17.14


In [42]:
statistical_analysis_summary = pd.DataFrame({
    "analysis": [
        "Matched competitive pricing",
        "Price rank consistency",
        "BFC vs Starbucks pricing",
        "BFC vs CBTL pricing",
        "Starbucks vs CBTL pricing",
        "Cross-brand individual ratings",
        "Customer experience themes"
    ],

    "method": [
        "Friedman test",
        "Kendall's W",
        "Exact Wilcoxon + Bonferroni",
        "Exact Wilcoxon + Bonferroni",
        "Exact Wilcoxon + Bonferroni",
        "Not tested",
        "Descriptive association analysis"
    ],

    "result": [
        "Q(2) = 10.000; p = 0.006738",
        "W = 1.000",
        "Raw p = 0.0625; adjusted p = 0.1875",
        "Raw p = 0.0625; adjusted p = 0.1875",
        "Raw p = 0.0625; adjusted p = 0.1875",
        "Inferentially ineligible",
        "Inferentially restricted"
    ],

    "interpretation": [
        "Systematic overall price-rank difference detected",
        "Complete rank consistency across 5 matched families",
        "BFC lower in 5/5; pairwise test not significant",
        "BFC lower in 5/5; pairwise test not significant",
        "Starbucks lower in 5/5; pairwise test not significant",
        "Comparable competitor individual ratings unavailable",
        "Patterns reported descriptively; no population-level inference"
    ]
})

statistical_analysis_summary

,analysis,method,result,interpretation
0,Matched competitive pricing,Friedman test,Q(2) = 10.000; p = 0.006738,Systematic overall price-rank difference detected
1,Price rank consistency,Kendall's W,W = 1.000,Complete rank consistency across 5 matched fam...
2,BFC vs Starbucks pricing,Exact Wilcoxon + Bonferroni,Raw p = 0.0625; adjusted p = 0.1875,BFC lower in 5/5; pairwise test not significant
3,BFC vs CBTL pricing,Exact Wilcoxon + Bonferroni,Raw p = 0.0625; adjusted p = 0.1875,BFC lower in 5/5; pairwise test not significant
4,Starbucks vs CBTL pricing,Exact Wilcoxon + Bonferroni,Raw p = 0.0625; adjusted p = 0.1875,Starbucks lower in 5/5; pairwise test not sign...
5,Cross-brand individual ratings,Not tested,Inferentially ineligible,Comparable competitor individual ratings unava...
6,Customer experience themes,Descriptive association analysis,Inferentially restricted,Patterns reported descriptively; no population...


In [43]:
research_question_status = pd.DataFrame({
    "research_question": [
        "RQ1 — Competitive Market Position",
        "RQ2 — Individual Customer Ratings",
        "RQ3 — Customer Experience Factors"
    ],

    "evidence_type": [
        "Descriptive + matched-price inference",
        "Descriptive only",
        "Descriptive association analysis"
    ],

    "statistical_status": [
        "Partially inferentially supported",
        "Cross-brand inference not eligible",
        "Population-level inference restricted"
    ],

    "key_finding": [
        (
            "BFC shows a consistent lower-price position "
            "across 5 matched product families; overall "
            "Friedman test significant."
        ),
        (
            "13 verified individual ratings were available "
            "only for BFC; cross-brand testing was not valid."
        ),
        (
            "Negative evidence was most associated with "
            "Order Execution; positive evidence was most "
            "associated with Product Quality."
        )
    ]
})

research_question_status

,research_question,evidence_type,statistical_status,key_finding
0,RQ1 — Competitive Market Position,Descriptive + matched-price inference,Partially inferentially supported,BFC shows a consistent lower-price position ac...
1,RQ2 — Individual Customer Ratings,Descriptive only,Cross-brand inference not eligible,13 verified individual ratings were available ...
2,RQ3 — Customer Experience Factors,Descriptive association analysis,Population-level inference restricted,Negative evidence was most associated with Ord...


In [44]:
price_matrix.to_csv(
    TABLES
    / "stat_matched_price_matrix.csv"
)

price_ranks.to_csv(
    TABLES
    / "stat_matched_price_ranks.csv"
)

pairwise_differences.to_csv(
    TABLES
    / "stat_pairwise_price_differences.csv",
    index=False
)

pairwise_results.to_csv(
    TABLES
    / "stat_pairwise_wilcoxon.csv",
    index=False
)

pairwise_business_summary.to_csv(
    TABLES
    / "stat_pairwise_business_summary.csv",
    index=False
)

bfc_individual_rating_summary.to_csv(
    TABLES
    / "stat_bfc_individual_rating_summary.csv",
    index=False
)

statistical_analysis_summary.to_csv(
    TABLES
    / "statistical_analysis_summary.csv",
    index=False
)

research_question_status.to_csv(
    TABLES
    / "research_question_status.csv",
    index=False
)

print(
    "Notebook 04 statistical outputs "
    "exported successfully."
)

Notebook 04 statistical outputs exported successfully.


# Statistical Analysis Conclusion

The statistical analysis provides inferential support for a systematic competitive pricing difference across the five directly matched product families.

The Friedman test produced:

\[
Q(2)=10.000,\quad p=0.006738
\]

leading to rejection of the null hypothesis at the 5% significance level.

Kendall's coefficient of concordance was:

\[
W=1.00
\]

showing complete consistency in the observed price ranking:

\[
\text{But First, Coffee}
<
\text{Starbucks}
<
\text{The Coffee Bean & Tea Leaf}
\]

across all five matched product families.

However, exact pairwise Wilcoxon tests did not reach statistical significance. Each comparison produced a raw two-sided p-value of 0.0625 and a Bonferroni-adjusted p-value of 0.1875. This reflects the limited statistical resolution of having only five matched product families and does not negate the observed directional consistency or commercial magnitude of the price gaps.

But First, Coffee was lower-priced than Starbucks in all five matched families by an average of ₱55 and lower-priced than The Coffee Bean & Tea Leaf by an average of ₱74.50.

Cross-brand individual-rating inference was not conducted because comparable individual-level ratings were unavailable for Starbucks and The Coffee Bean & Tea Leaf. The 13 verified individual ratings available for But First, Coffee were retained as descriptive evidence only.

Customer-experience themes were also treated descriptively because the review sample is observational, clustered across branches, multi-theme, and only partially classifiable using the conservative rule-based polarity method.

Overall, the statistical evidence supports a consistent affordability position for But First, Coffee within the matched competitive sample. It does not establish that broad price increases would improve revenue or profitability.

The next stage therefore shifts from statistical comparison to revenue-opportunity analysis, evaluating how pricing, promotions, product mix, customer experience, and commercial access could contribute to profitable revenue growth while preserving the brand's affordability position.